# QuantOne v2 — Phase 0 GPU probe

For each manifest model: download its SMALLEST study quant, verify sha256, load with llama-cpp-python 0.3.33, apply embedded chat template, assert thinking stays off, run 10 sample tasks, measure tok/s. Writes `phase0_gpu_report.json`.

Small tier → Kaggle T4. Big tier (qwen36_27b, gemma4_26b_a4b) → Colab L4 (set `TIER = "big"` there). MoE memory for gemma4_26b_a4b gets measured here, not assumed.

In [ ]:
TIER = "small"   # "small" (Kaggle T4) or "big" (Colab L4/A100)
REPO_GIT_URL = "https://github.com/kaushiksai29/QuantOne.git"
BRANCH = "v2"

In [ ]:
import os, shutil, subprocess

def gpu_name():
    if shutil.which("nvidia-smi") is None:
        return None
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None

if not os.path.exists("QuantOne"):
    !git clone -b {BRANCH} {REPO_GIT_URL}
%cd QuantOne

GPU = gpu_name()
if GPU is None:
    raise SystemExit(
        "STOP: no GPU in this session. This is the GPU probe — its tok/s feed "
        "the Phase 4 GPU-hour budget, so CPU numbers would mislead it. Turn on "
        "the accelerator (Kaggle: right panel -> Accelerator -> GPU T4; Colab: "
        "Runtime -> Change runtime type -> T4/L4), then Run All. If it was "
        "already on, do Run -> Factory reset to clear the bad install first.")
print("GPU detected:", GPU)

# GPU present -> CUDA wheel; source-build fallback if the prebuilt won't load
!pip install -q llama-cpp-python==0.3.33 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 PyYAML
try:
    import llama_cpp
except Exception as exc:
    print(f"prebuilt CUDA wheel unusable ({exc}); building from source (~15-25 min) ...")
    !pip uninstall -y -q llama-cpp-python
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    !pip install -q llama-cpp-python==0.3.33 --no-binary llama-cpp-python
    import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU:", GPU)

In [ ]:
import json, hashlib, re, time, urllib.request, os
from pathlib import Path
from llama_cpp import Llama

OUTDIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "."
manifest = json.load(open("manifest.json"))
models = {k: m for k, m in manifest["models"].items() if m["tier"] == TIER}
tasks = [json.loads(l) for l in open("tasks/tasks.jsonl")][:10]
THINK_RE = re.compile(r"<think|<thinking|</think", re.I)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def gen(llm, messages, max_tokens=256):
    out = llm.create_chat_completion(messages=messages, temperature=0,
                                     max_tokens=max_tokens, seed=0)
    return out["choices"][0]["message"]["content"], out["usage"]["completion_tokens"]

report = []
for key, m in models.items():
    quant, info = sorted(m["files"].items(), key=lambda kv: kv[1]["gb"] if isinstance(kv[1], dict) else 9e9)[0]
    url = f"https://huggingface.co/{m['gguf_repo']}/resolve/main/{info['path']}"
    local = Path(info["path"].split("/")[-1])
    print(f"== {key} ({quant}, {info['gb']}GB) ==", flush=True)
    if not local.exists():
        urllib.request.urlretrieve(url, local)
    actual = sha256_file(local)
    assert actual == info["sha256"], f"sha mismatch for {key}: {actual}"
    entry = {"model": key, "quant": quant, "sha256": "OK"}
    try:
        llm = Llama(model_path=str(local), n_ctx=4096, chat_format=None,
                    n_gpu_layers=-1, verbose=False)
        entry["load"] = "OK"

        # --- thinking-off mechanism discovery on the first task ---
        txt, _ = gen(llm, [{"role": "user", "content": tasks[0]["prompt"]}])
        if not THINK_RE.search(txt):
            entry["thinking_off"] = "none needed (no think tokens by default)"
            sys_prefix = []
        else:
            for mech, sys_prefix in [
                ("/no_think system prompt",
                 [{"role": "system", "content": "/no_think"}]),
                ("'/no_think' in user turn", None),
            ]:
                if sys_prefix is None:
                    t2, _ = gen(llm, [{"role": "user",
                                       "content": tasks[0]["prompt"] + " /no_think"}])
                else:
                    t2, _ = gen(llm, sys_prefix + [{"role": "user",
                                                    "content": tasks[0]["prompt"]}])
                if not THINK_RE.search(t2):
                    entry["thinking_off"] = mech
                    break
            else:
                entry["thinking_off"] = "UNRESOLVED — kill or fix before Phase 3"
                sys_prefix = []

        # --- 10 tasks with the working mechanism: tok/s + leak check ---
        t0 = time.perf_counter(); n_tok = 0; thinks = 0; parses = 0
        msgs_prefix = sys_prefix if isinstance(sys_prefix, list) else []
        for t in tasks:
            txt, ct = gen(llm, msgs_prefix + [{"role": "user", "content": t["prompt"]}])
            n_tok += ct
            thinks += bool(THINK_RE.search(txt))
            parses += txt.strip().startswith(("{", "`"))
        dt = time.perf_counter() - t0
        entry.update(tok_per_s=round(n_tok/dt, 1), think_leaks=f"{thinks}/10",
                     json_shaped=f"{parses}/10")
        del llm
    except Exception as e:
        entry["load"] = f"FAIL: {e}"
    print("  ", entry, flush=True)
    report.append(entry)
    local.unlink()  # disk budget

# GPU name is captured in cell-2 as `GPU`; tok/s is meaningless without it.
out = {"meta": {"tier": TIER, "gpu": GPU, "llama_cpp": llama_cpp.__version__},
       "models": report}
out_path = f"{OUTDIR}/phase0_gpu_report_{TIER}.json"
json.dump(out, open(out_path, "w"), indent=2)
print(f"\nDONE on {GPU} — report at {out_path}")